# GraspMAS + GraspGen-X — 6-DoF language-driven grasping

This notebook replaces the original planar demo. `grasp_detection` no longer returns a
rotated rectangle `[quality, x, y, w, h, angle]` — it returns a full **6-DoF SE(3) grasp
pose**: a 3D position in metres plus the direction the gripper approaches from and the
direction its fingers close along.

**Conventions used throughout** (see `CLAUDE.md` §5):
- Units are **metres**, poses are in the **camera frame**.
- A pose is anchored at the **gripper base**: `pose[:3, 2]` (+Z) is the approach axis,
  `pose[:3, 0]` (+X) is the closing direction.

## Before you run this

1. **Start the GraspGen-X server** (it lives in the other conda env and holds the model):
   ```bash
   scripts/run_server.sh --daemon
   ```
2. **Set a free LLM key** for the agent loop — only needed for Part 2.
   Get one at <https://aistudio.google.com/apikey>, then `export LLM_API_KEY=...`
   or write it to `GraspMAS/api.key`.
   *Parts 1 and 3 need no key: grounding runs on local GroundingDINO / SAM / VLPart weights.*
3. Run this notebook with the **`graspmas`** kernel.

In [ ]:
import json, sys, warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")
%matplotlib inline

REPO = Path.cwd().parent if Path.cwd().name == "GraspMAS" else Path.cwd()
sys.path.insert(0, str(REPO / "GraspMAS"))

# Point at the shipped GraspGen-X sample scene: real RGB, real metric depth and
# real intrinsics, so nothing here depends on guessed calibration.
SCENE = REPO / "GraspGenX" / "assets" / "sample_data" / "real_world" / "00"
rgb = cv2.imread(str(SCENE / "rgb.png"))[:, :, ::-1].copy()
depth = np.load(SCENE / "depth.npy").astype(np.float32)          # metres
K = np.asarray(json.loads((SCENE / "meta_data.json").read_text())["intrinsics"])

print("rgb  ", rgb.shape)
print("depth", depth.shape, f"{depth[depth>0].min():.2f}-{depth.max():.2f} m")
print("K\n", np.round(K, 1))

plt.figure(figsize=(11, 6))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.axis("off"); plt.title("RGB")
d = np.where(depth > 0, depth, np.nan)
plt.subplot(1, 2, 2); plt.imshow(d, cmap="turbo"); plt.axis("off"); plt.title("depth (m)")
plt.tight_layout()

## Part 1 — The grasp path on its own (no LLM key needed)

`find()` uses local GroundingDINO + SAM, so text → mask → metric point cloud →
GraspGen-X → 6-DoF pose runs entirely offline. This is the part of the system the
upgrade actually replaced.

In [ ]:
import image_patch as ip
from perception3d import SceneContext, approach_elevation_deg
from vis6d import visualize_grasp_6dof

# Check the server is up before doing anything expensive.
client = ip.get_graspgen_client()
assert client.is_alive(), "Start the server first:  scripts/run_server.sh --daemon"
print("server:", client.metadata["model"])

# Publish the calibration for this image. The LLM-generated code only ever
# receives the RGB array, so depth + K reach grasp_detection through module state.
ip.configure_scene(
    scene=SceneContext(depth=depth, intrinsics=K, image_shape=rgb.shape[:2]),
    gripper_name="franka_panda",
    num_grasps=100,
    planner="graspmoe",
)

root = ip.ImagePatch(rgb)

In [ ]:
TARGET = "mustard bottle"

patches = root.find(TARGET)
print(f"grounded {len(patches)} candidate(s) for {TARGET!r}")

grasp = root.grasp_detection(patches[0])
assert grasp is not None, "no grasp produced"

print(f"\ngripper  : {grasp['gripper']}")
print(f"score    : {grasp['score']:.3f}")
print(f"position : {np.round(grasp['position'], 4)} m  (camera frame)")
print(f"approach : {np.round(grasp['approach'], 3)}"
      f"   -> {approach_elevation_deg(grasp['approach']):.1f}deg off the optical axis")
print(f"closing  : {np.round(grasp['closing'], 3)}")
print(f"jaw width: {grasp['width']*100:.1f} cm")
print("\npose (4x4):")
print(np.round(np.asarray(grasp["pose"]), 4))

In [ ]:
# The projected gripper: orange = fingers, yellow = the line they close along,
# blue = approach axis, magenta = the region the language selected.
# This is exactly the image the Observer agent is shown.
out = visualize_grasp_6dof(rgb, grasp, K, save_folder="/tmp/graspmas_demo",
                           filename="demo.png", mask=patches[0].mask)
plt.figure(figsize=(13, 7))
plt.imshow(cv2.imread(str(out))[:, :, ::-1]); plt.axis("off")
plt.title(f"6-DoF grasp on the {TARGET}")

### Why this is more than the old rectangle

The planar model emitted `[quality, x, y, w, h, angle]` — an image-plane box. The
ManiSkill demo then had to *invent* the missing degrees of freedom, hardcoding a
straight-down approach (`approaching = [0, 0, -1]`) and taking only the in-plane
rotation from the rectangle. That is 4 DoF wearing a 6-DoF label.

Here the approach direction is predicted from the object's 3D geometry, so a bottle
lying on its side and a bottle standing upright get genuinely different hand poses.

In [ ]:
# The full candidate set, coloured by discriminator score (red low -> green high).
from vis6d import visualize_grasp_candidates
from perception3d import unproject, downsample_cloud

cloud = downsample_cloud(unproject(depth, K, patches[0].mask.astype(bool)))
grasps, scores = client.infer(cloud, gripper_name="franka_panda",
                              num_grasps=100, planner="graspmoe")
print(f"{len(grasps)} candidates, scores {scores.min():.2f}-{scores.max():.2f}")

cand = visualize_grasp_candidates(rgb, grasps, scores, K,
                                  save_folder="/tmp/graspmas_demo")
plt.figure(figsize=(13, 7))
plt.imshow(cv2.imread(str(cand))[:, :, ::-1]); plt.axis("off")

elev = [approach_elevation_deg(g[:3, 2]) for g in grasps]
plt.figure(figsize=(7, 3))
plt.hist(elev, bins=24, color="#2a78d6")
plt.axvline(0, color="#eb6834", ls="--", lw=2)
plt.xlabel("approach angle off the optical axis (deg)")
plt.ylabel("candidates")
plt.title("Approach directions the model considered\n(orange = the single direction the 2D pipeline could express)")
plt.tight_layout()

## Part 2 — Multi-embodiment

One model, any gripper. The gripper's swept volume is what conditions the diffusion
model, so switching hands needs no retraining and no server restart.

In [ ]:
for gripper in ["franka_panda", "robotiq_2f_85", "unitree_g1"]:
    ip._GRASP_CACHE.clear()
    g = root.grasp_detection(patches[0], gripper_name=gripper)
    if g is None:
        print(f"{gripper:16s} no grasp")
        continue
    print(f"{gripper:16s} score={g['score']:.3f}  jaw={g['width']*100:5.1f}cm  "
          f"approach={approach_elevation_deg(g['approach']):5.1f}deg")
    visualize_grasp_6dof(rgb, g, K, save_folder="/tmp/graspmas_demo",
                         filename=f"{gripper}.png", mask=patches[0].mask)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, gripper in zip(axes, ["franka_panda", "robotiq_2f_85", "unitree_g1"]):
    p = Path("/tmp/graspmas_demo") / f"{gripper}.png"
    if p.exists():
        ax.imshow(cv2.imread(str(p))[:, :, ::-1])
    ax.set_title(gripper); ax.axis("off")
plt.tight_layout()

## Part 3 — The full closed loop (needs a free LLM key)

Planner → Coder → Observer. The Observer is a *vision* model: it looks at the rendered
grasp above and critiques it, and its summary feeds the next Planner round.

Free-tier budget: roughly 3 LLM calls per round, so `max_round=2` here keeps one query
to about 6-10 requests.

In [ ]:
import asyncio
from agents.graspmas import GraspMAS
from run_artifacts import RunRecorder

recorder = RunRecorder(name="notebook_demo")
recorder.write_config({"query": "grasp the mustard bottle", "notebook": True})

graspmas = GraspMAS(
    api_file="api.key",          # or export LLM_API_KEY
    max_round=2,
    gripper_name="franka_panda",
    num_grasps=100,
    recorder=recorder,
)

out, grasp_pose = await graspmas.query(
    "Grasp the mustard bottle.",
    rgb,
    save_folder=str(recorder.images_dir),
    depth=depth,
    intrinsics=K,
)

print("\nfinal grasp:", None if grasp_pose is None else
      {k: np.round(grasp_pose[k], 3).tolist() if isinstance(grasp_pose[k], list)
       else grasp_pose[k] for k in ("gripper", "score", "position", "approach")})
print("LLM calls used:", recorder.llm_call_count)
print("artifacts:", recorder.dir)

### Part-level grasping — the reason the multi-agent loop is worth keeping

The point of GraspMAS is that language reaches the *sampler*, not just object selection.
`find_part` returns a mask of the named part, and because that mask is what becomes the
point cloud, the 6-DoF model plans on the part itself.

In [ ]:
out, grasp_pose = await graspmas.query(
    "Grasp the toy airplane by its wing.",
    rgb,
    save_folder=str(recorder.images_dir),
    depth=depth,
    intrinsics=K,
)
print(grasp_pose["position"] if grasp_pose else "no grasp")

---

## Where things are

| | |
|---|---|
| Architecture, invariants, gotchas | `CLAUDE.md` |
| Robotics evaluation of the result | `SUMMARY.md` |
| Run artifacts and their schema | `outputs/README.md` |
| Offline test suite (no key, no server) | `scripts/run_tests.sh` |
| CPU latency benchmark | `scripts/bench_cpu.py` |
| Full pipeline check without an LLM | `scripts/verify_pipeline.py` |